# E18A · Early-K / Late-C 4-State Diagnostic
**[해결책] 원래 K/C 4-state 분류를 유지한 개선 검증.**

E17처럼 context/closed-book 두 분기를 만들지 않는다. **하나의 원래 Q-first 질문·문맥 입력**에서 L7/L13 및 질문 끝/프롬프트 끝의 표현을 읽는다.

**이번 단계는 supervised 진단 probe를 사용한다.** 답변 생성, steering, probe-free routing까지 성공했다는 실험이 아니다. LLM 가중치는 고정한다.

**실행 상태:** 전체 early/late 실험 결과는 비어 있다. 실제 L13 bank로 실행한 M0/M1 사전 점검은 패키지 `local_validation`에 별도 제공한다.

## 1. 문제상황 → 원인 가설 → 비교
기존 코드에서도 K/C 이진 probe와 질문/마지막 위치 분리가 있었다. 따라서 단순히 이진 분류기를 두 개 만드는 것 자체가 새 아이디어는 아니다.

검증할 가설은 **“K를 읽는 레이어를 L13에서 L7로 바꾸면, 같은 분류 구조에서도 4-state가 더 잘 읽히는가?”**이다. 질문 위치와 마지막 위치를 바꾸는 효과를 별도로 통제한다.

|ID|입력/읽는 방식|목적|
|---|---|---|
|M0|L13 마지막 → 직접 4-class|기존 full-hidden 기준선|
|M1|K:L13 마지막 / C:L13 마지막|factorization 대조군|
|M2|K:L7 마지막 / C:L13 마지막|**주 실험: early/late**|
|M3|K:L13 질문 끝 / C:L13 마지막|위치만 변경|
|M4|K:L7 질문 끝 / C:L13 마지막|레이어+위치 변경|
|M5|[L7 질문 끝, L13 마지막] → 직접 4-class|두 벡터 공동 판별|
|M6|[L13 질문 끝, L13 마지막] → 직접 4-class|M5와 차원 수 동일한 대조군|

**주 비교 M2−M1.** M2−M0는 실용적 기준선 비교다. M5를 M4와 비교할 때는 모델 파라미터 수 차이도 있으므로 ‘정보 보존만의 인과 효과’라고 결론내리지 않는다.

## 2. 설정
새 Colab 런타임에서 실행한다. 기존 cache 경로가 맞으면 모델 가중치를 로딩할 필요가 없다. cache가 없으면 GPU에서 중립 prefill만 수집한다.

- `MODE='full'`: 전체 7조건. **최종 기본값**.
- `MODE='l13_only'`: M0/M1 점검만 한다. early/late 가설은 검사하지 않는다.
- `FEATURE_SOURCE='auto'`: 완전한 E11 cache가 있으면 재사용, 없으면 새로 수집.
- 원본 cache가 발견됐는데 검증을 실패하면 자동으로 다른 데이터로 바꾸지 않고 중단한다.
- 기존 실험 폴더는 읽기만 한다. 출력은 새로운 E18A 폴더에만 저장한다.

In [ ]:
import sys, os, json, gc, importlib.util
from pathlib import Path
IN_COLAB = importlib.util.find_spec('google.colab') is not None
RUN_ID = 'E18A_earlyK_lateC_v1'
MODE = 'full'  # 'full' 또는 의도적인 'l13_only'
FEATURE_SOURCE = 'auto'  # 'auto', 'e11_cache', 'fresh'
INSTALL_DEPENDENCIES = True
REQUIRE_ORIGINAL_SKLEARN = True
BATCH_SIZE = 4
BOOTSTRAP_REPS = 2000
SAVE_MODELS = False
PROJECT_ROOT = Path('/content/drive/MyDrive/p8s') if IN_COLAB else Path('/mnt/data')
E11_NEUTRAL = PROJECT_ROOT / 'E11_run01/fold_0/e11-qfirst/neutral'
E14_ROOT = PROJECT_ROOT / 'E14_L13_head_5fold_v1'
assert MODE in {'full', 'l13_only'}
assert FEATURE_SOURCE in {'auto', 'e11_cache', 'fresh'}

In [ ]:
# 환경을 고정한다. CUDA torch wheel을 임의로 교체하지 않는다.
import subprocess
if IN_COLAB and INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'scikit-learn==1.6.1', 'transformers==4.57.1',
        'accelerate', 'safetensors', 'huggingface-hub',
        'google-api-python-client', 'threadpoolctl', 'pytest'])
import sklearn
if REQUIRE_ORIGINAL_SKLEARN and sklearn.__version__ != '1.6.1':
    raise RuntimeError('E14 기준 sklearn==1.6.1 필요. 설치 뒤 세션을 다시 시작하고 처음부터 실행하세요.')
print('sklearn:', sklearn.__version__)
if not REQUIRE_ORIGINAL_SKLEARN:
    print('주의: 원래 라이브러리 버전과 다를 수 있습니다. baseline 재현 감사 결과를 확인하세요.')

In [ ]:
#@title 내부 구현 (수정 없이 실행)
# 소스가 내장되어 있어 ipynb 하나만 업로드하면 됩니다.
MODULE_DIR = (Path('/content/e18a_modules') if IN_COLAB else Path.cwd() / 'e18a_modules')
MODULE_DIR.mkdir(parents=True, exist_ok=True)
_MODULE_SOURCES = {'e18a_core.py': '"""E18A: fixed-design, question-grouped K/C four-state diagnostic evaluation.\n\nThis module does not modify LLM weights, perform QA generation, or implement steering.\nAll learned readers are supervised diagnostic probes, not label-free routers.\n"""\nfrom __future__ import annotations\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport time\nimport warnings\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Mapping\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,\n                             f1_score, log_loss, precision_recall_fscore_support, roc_auc_score)\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\n\nSTATE_NAMES = [\'K+C+\', \'K+C-\', \'K-C-\', \'K-C+\']\nMODEL_ID = \'meta-llama/Llama-3.1-8B-Instruct\'\nMODEL_REVISION = \'0e9e39f249a16976918f6564b8830bc894c89659\'\nSYSTEM_PROMPT = (\n    \'You are a question answering system. Answer the question based on the given context. \'\n    \'Output the answer string itself and nothing else: no sentence, no explanation, \'\n    \'no punctuation, no quotes. Keep it as short as possible (a name or a term).\'\n)\nSOURCE_FILES = {\n    \'behavior\': dict(id=\'1RXvW6o552ZNZn1PUMRaf8CEDMIUpqQOr\', name=\'p8s_behavior_v2.csv\',\n        sha256=\'fd37b0afd04972e3834ffa5f0af7f362affcfac8e20325117d37d8de15447c1c\'),\n    \'bank13\': dict(id=\'1GK0xi217u57Wz_kn8ycg5eSlPgjsaFNy\', name=\'L13_neutral_head_bank_all_rows.npz\',\n        sha256=\'4ad4a040388d1e45995be7aadc7b91f6dfd1f4091b4fd452da8b4a9036b83802\'),\n    \'oof13\': dict(id=\'1djoU4xPIAnCQjBLrb4a3YYIqStWLM6En\', name=\'OOF_L13_LAST_HIDDEN_PREDICTIONS.csv\',\n        sha256=\'2f8f214aa44fe1a6e020edbb1bf3456f9ed30aa147d9bce09b30bfa72150677d\'),\n}\n\n@dataclass(frozen=True)\nclass Spec:\n    name: str\n    mode: str\n    keys: tuple[str, ...]\n    purpose: str\n\nSPECS = (\n    Spec(\'M0_L13_joint4\', \'joint4\', (\'L13\',), \'E14 L13-last joint four-class reference\'),\n    Spec(\'M1_L13_factor\', \'factor\', (\'L13\', \'L13\'), \'Same-layer K/C factorization control\'),\n    Spec(\'M2_L7lastK_L13lastC\', \'factor\', (\'L7\', \'L13\'), \'PRIMARY: change K reading layer only\'),\n    Spec(\'M3_L13qK_L13lastC\', \'factor\', (\'Q13\', \'L13\'), \'Change K token location only\'),\n    Spec(\'M4_L7qK_L13lastC\', \'factor\', (\'Q7\', \'L13\'), \'Early K plus question-position reading\'),\n    Spec(\'M5_L7q_L13last_joint4\', \'joint4\', (\'Q7\', \'L13\'), \'Two-vector joint decoder\'),\n    Spec(\'M6_L13q_L13last_joint4\', \'joint4\', (\'Q13\', \'L13\'), \'Dimension-matched control for M5\'),\n)\nCONTRASTS = (\n    (\'M2_L7lastK_L13lastC\', \'M1_L13_factor\', \'PRIMARY_layer_only\'),\n    (\'M2_L7lastK_L13lastC\', \'M0_L13_joint4\', \'secondary_vs_original\'),\n    (\'M1_L13_factor\', \'M0_L13_joint4\', \'factorization_only\'),\n    (\'M3_L13qK_L13lastC\', \'M1_L13_factor\', \'position_only\'),\n    (\'M4_L7qK_L13lastC\', \'M3_L13qK_L13lastC\', \'early_layer_at_question\'),\n    (\'M5_L7q_L13last_joint4\', \'M6_L13q_L13last_joint4\', \'joint_dimension_matched_layer\'),\n    (\'M5_L7q_L13last_joint4\', \'M4_L7qK_L13lastC\', \'joint_vs_factor_not_parameter_matched\'),\n)\n\ndef sha256_file(path: str | Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef write_json(path: str | Path, obj) -> None:\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.tmp\')\n    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef environment() -> dict:\n    out = {\'python\': platform.python_version(), \'platform\': platform.platform()}\n    for name in [\'numpy\', \'pandas\', \'scikit-learn\', \'torch\', \'transformers\', \'threadpoolctl\']:\n        try: out[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError: out[name] = None\n    return out\n\n\ndef freeze_plan(out_dir: str | Path, *, specs=SPECS, bootstrap=2000, seed=42) -> dict:\n    """A timestamped local analysis lock, NOT independent/external preregistration."""\n    p = Path(out_dir) / \'analysis_plan.json\'\n    plan = {\n        \'experiment\': \'E18A_EarlyK_LateC_4State_v1\',\n        \'status\': \'development_followup_not_independent_confirmation\',\n        \'split\': \'E14 fixed question-grouped 5 outer folds, 60% fit/10% UF/10% PC/20% test\',\n        \'state_order\': STATE_NAMES, \'K_definition\': \'E14 kside_corr; DO NOT use E17 K_strict\',\n        \'model_id\': MODEL_ID, \'model_revision\': MODEL_REVISION,\n        \'prompt\': \'original E11/E14 e11-qfirst, one question/context prompt per row\',\n        \'layer_indexing\': \'zero-based decoder block output: layers[7], layers[13]\',\n        \'C_logreg\': 0.1, \'solver\': \'lbfgs\', \'max_iter\': 2000, \'seed\': seed,\n        \'standardization\': \'fit partition only; no test normalization\',\n        \'binary_threshold\': 0.5, \'class_weight\': None,\n        \'methods\': [asdict(x) for x in specs], \'contrasts\': list(CONTRASTS),\n        \'bootstrap_replicates\': bootstrap,\n        \'bootstrap_unit\': \'question; both C conditions together; conditional on fitted OOF predictions\',\n        \'primary_outcome\': \'M2-M1 pooled OOF 4-state macro-F1; paired question CI\',\n        \'no_claims\': [\'no QA EM improvement measured\', \'no causal intervention tested\',\n                      \'not runtime probe-free\', \'no guaranteed improvement\',\n                      \'no independent held-out confirmation\'],\n    }\n    canonical = json.loads(json.dumps(plan))\n    if p.exists():\n        old = json.loads(p.read_text(encoding=\'utf-8\'))\n        old.pop(\'created_utc\', None)\n        if old != canonical:\n            raise ValueError(\'Analysis plan differs. Use a NEW run directory; do not overwrite results.\')\n    else:\n        write_json(p, {**canonical, \'created_utc\': time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime())})\n    (p.parent / \'analysis_plan.sha256\').write_text(sha256_file(p) + \'\\n\')\n    return canonical\n\n\ndef cell_ids(k, c) -> np.ndarray:\n    k, c = np.asarray(k, int), np.asarray(c, int)\n    if not np.isin(k, [0, 1]).all() or not np.isin(c, [0, 1]).all():\n        raise ValueError(\'K/C must be binary.\')\n    return np.where(k == 1, np.where(c == 1, 0, 1), np.where(c == 1, 3, 2)).astype(int)\n\n\ndef load_labels(behavior: str | Path, oof_reference: str | Path | None = None) -> pd.DataFrame:\n    f = pd.read_csv(behavior).reset_index(drop=True)\n    for name in [\'question\', \'ctx_text\', \'ctx_label\', \'kside_corr\']:\n        if name not in f or f[name].isna().any(): raise ValueError(f\'Missing/empty column: {name}\')\n    f[\'k\'] = f.kside_corr.map({\'K+\': 1, \'K-\': 0, \'K−\': 0})\n    f[\'c\'] = f.ctx_label.map({\'C+\': 1, \'C-\': 0, \'C−\': 0})\n    if f[[\'k\', \'c\']].isna().any().any(): raise ValueError(\'Unknown K/C encoding; no silent relabeling.\')\n    f[[\'k\', \'c\']] = f[[\'k\', \'c\']].astype(int)\n    f[\'state\'] = cell_ids(f.k, f.c); f[\'row\'] = np.arange(len(f))\n    if len(f) != 1224 or f.question.nunique() != 612: raise ValueError(\'Expected original E14 612Q / 1224 rows.\')\n    if not f.groupby(\'question\').size().eq(2).all(): raise ValueError(\'Question pair size must be 2.\')\n    if not f.groupby(\'question\').k.nunique().eq(1).all(): raise ValueError(\'K inconsistent across a pair.\')\n    if not f.groupby(\'question\').c.nunique().eq(2).all(): raise ValueError(\'Need exactly C+ and C- per question.\')\n    if oof_reference is not None:\n        r = pd.read_csv(oof_reference).sort_values(\'row\')\n        if not np.array_equal(r.row, f.row) or not np.array_equal(r.true_state, f.state):\n            raise ValueError(\'E14 stored labels disagree with kside_corr. Stop and reconcile; never replace labels.\')\n        fold = f.question.map(build_outer_map(f)).to_numpy()\n        if not np.array_equal(r.fold, fold): raise ValueError(\'E14 outer fold mismatch.\')\n    return f\n\n\ndef build_outer_map(f: pd.DataFrame, seed=42) -> dict:\n    q = f[[\'question\', \'k\']].drop_duplicates().sort_values(\'question\').reset_index(drop=True)\n    rng = np.random.default_rng(seed); mapping = {}\n    for cls in (0, 1):\n        ids = q.loc[q.k.eq(cls), \'question\'].to_numpy().copy(); rng.shuffle(ids)\n        mapping.update({s: i % 5 for i, s in enumerate(ids)})\n    return mapping\n\n\ndef make_split(f: pd.DataFrame, test_fold: int, seed=42) -> pd.DataFrame:\n    fold = f.question.map(build_outer_map(f, seed)).to_numpy()\n    role = np.full(len(f), \'fit\', object)\n    role[fold == test_fold] = \'test\'; role[fold == (test_fold + 1) % 5] = \'cal\'\n    rng = np.random.default_rng(seed + 1000 * test_fold)\n    for cls in (0, 1):\n        qids = np.array(sorted(set(f.loc[(role == \'cal\') & f.k.eq(cls), \'question\'])))\n        rng.shuffle(qids); half = len(qids) // 2\n        role[(role == \'cal\') & f.question.isin(qids[:half])] = \'utility_fit\'\n        role[(role == \'cal\') & f.question.isin(qids[half:])] = \'policy_cal\'\n    sp = f[[\'row\', \'question\', \'k\', \'c\', \'state\']].copy()\n    sp[\'outer_fold\'] = fold; sp[\'role\'] = role\n    if not sp.groupby(\'question\').role.nunique().eq(1).all(): raise ValueError(\'Group leakage.\')\n    for r in [\'fit\', \'utility_fit\', \'policy_cal\', \'test\']:\n        if set(sp.loc[sp.role.eq(r), \'state\']) != {0, 1, 2, 3}:\n            raise ValueError(f\'Missing state in fold {test_fold}, role {r}.\')\n    return sp\n\n\ndef factor_probs(pk, pc) -> np.ndarray:\n    pk = np.asarray(pk, float); pc = np.asarray(pc, float)\n    if pk.shape != pc.shape or pk.ndim != 1: raise ValueError(\'Use matching 1-D probabilities.\')\n    if not np.isfinite(pk).all() or not np.isfinite(pc).all(): raise ValueError(\'Nonfinite probability.\')\n    if ((pk < 0) | (pk > 1) | (pc < 0) | (pc > 1)).any(): raise ValueError(\'Not probabilities.\')\n    return np.stack([pk * pc, pk * (1 - pc), (1 - pk) * (1 - pc), (1 - pk) * pc], axis=1)\n\n\ndef validate_features(features: Mapping[str, np.ndarray], n: int) -> None:\n    for k, x in features.items():\n        if x.ndim != 2 or len(x) != n or not np.isfinite(x).all():\n            raise ValueError(f\'Invalid feature {k}: {x.shape}\')\n\n\ndef load_e14_block_bank(path: str | Path, n: int) -> dict[str, np.ndarray]:\n    with np.load(path, allow_pickle=False) as z:\n        rows = z[\'rows\'].astype(int)\n        if not np.array_equal(np.sort(rows), np.arange(n)): raise ValueError(\'Invalid bank rows.\')\n        order = np.argsort(rows)\n        return {\'L13\': z[\'BLOCK_LAST\'][order].astype(np.float32)}\n\n\ndef render_prompt(tokenizer, question: str, context: str) -> dict:\n    user = f\'Question: {question}\\nContext: {context}\\nAnswer:\'\n    text = tokenizer.apply_chat_template([{\'role\': \'system\', \'content\': SYSTEM_PROMPT},\n        {\'role\': \'user\', \'content\': user}], tokenize=False, add_generation_prompt=True)\n    base = text.rfind(user)\n    if base < 0: raise ValueError(\'Chat template changed user text.\')\n    qstart = base + len(\'Question: \'); qend = qstart + len(question)\n    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)\n    qpos = [i for i, (s, e) in enumerate(enc[\'offset_mapping\']) if e > qstart and s < qend and e > s]\n    if not qpos: raise ValueError(\'Question token span not found.\')\n    return {\'ids\': enc[\'input_ids\'], \'qpos\': qpos[-1],\n            \'prompt_sha256\': hashlib.sha256(text.encode()).hexdigest()}\n\n\ndef load_e11_neutral(folder: str | Path, f: pd.DataFrame, tokenizer=None) -> tuple[dict, dict]:\n    """Load ONLY complete neutral Q-first shards; verify semantics and row identity."""\n    folder = Path(folder)\n    manifest_path = folder / \'manifest.json\'\n    if not manifest_path.exists(): raise FileNotFoundError(manifest_path)\n    meta = json.loads(manifest_path.read_text())\n    expected = {\'rows\': list(range(len(f))), \'protocol\': \'e11-qfirst\', \'directive\': None,\n                \'collect\': True, \'batch_size\': 4, \'tag\': \'neutral\'}\n    if meta != expected: raise ValueError(\'Not the original complete Q-first neutral manifest.\')\n    if tokenizer is None: raise ValueError(\'Tokenizer required to verify every cached prompt hash.\')\n    chunks = {k: [] for k in [\'Q7\', \'L7\', \'Q13\', \'L13\']}; allrows = []; hashes = []\n    for start in range(0, len(f), 4):\n        npz = folder / f\'batch_{start:06d}.npz\'; js = npz.with_suffix(\'.json\')\n        if not npz.exists() or not js.exists(): raise FileNotFoundError(f\'Incomplete neutral cache: {npz}\')\n        rec = json.loads(js.read_text())\n        with np.load(npz, allow_pickle=False) as z:\n            rows = z[\'rows\'].astype(int)\n            if not np.array_equal(rows, np.arange(start, min(start + 4, len(f)))):\n                raise ValueError(\'Unexpected shard row ids.\')\n            if [int(r[\'row\']) for r in rec] != rows.tolist(): raise ValueError(\'npz/json row mismatch.\')\n            for r in rec:\n                i = int(r[\'row\'])\n                if r[\'action\'] != \'noop\' or float(r[\'max_applied_delta_ratio\']) != 0:\n                    raise ValueError(\'Steered features must not enter neutral state diagnostics.\')\n                want = render_prompt(tokenizer, str(f.iloc[i].question), str(f.iloc[i].ctx_text))[\'prompt_sha256\']\n                if r[\'prompt_sha256\'] != want: raise ValueError(f\'Cached prompt differs for row {i}.\')\n            for key in chunks: chunks[key].append(z[key].astype(np.float32))\n            allrows.extend(rows.tolist())\n        hashes.append({\'name\': npz.name, \'sha256\': sha256_file(npz), \'json_sha256\': sha256_file(js)})\n    if allrows != list(range(len(f))): raise ValueError(\'Incomplete/duplicate features.\')\n    features = {k: np.concatenate(v) for k, v in chunks.items()}\n    validate_features(features, len(f))\n    return features, {\'source\': \'E11_original_neutral\', \'manifest_sha256\': sha256_file(manifest_path),\n                      \'all_prompt_hashes_verified\': True, \'shards\': hashes}\n\n\ndef feature_agreement(x, ref) -> dict:\n    x = np.asarray(x, np.float64); ref = np.asarray(ref, np.float64)\n    if x.shape != ref.shape: raise ValueError(\'Feature bank dimension mismatch.\')\n    rel = np.linalg.norm(x - ref, axis=1) / np.maximum(np.linalg.norm(ref, axis=1), 1e-12)\n    cos = np.sum(x * ref, axis=1) / np.maximum(np.linalg.norm(x, axis=1) * np.linalg.norm(ref, axis=1), 1e-12)\n    return {\'max_abs_difference\': float(np.max(np.abs(x - ref))),\n            \'median_relative_L2\': float(np.median(rel)), \'max_relative_L2\': float(rel.max()),\n            \'median_cosine\': float(np.median(cos)), \'min_cosine\': float(cos.min())}\n\n\ndef question_invariance(f, features) -> dict:\n    pairs = [g.row.to_numpy() for _, g in f.groupby(\'question\', sort=True)]\n    a = np.array([p[0] for p in pairs]); b = np.array([p[1] for p in pairs])\n    return {k: feature_agreement(x[a], x[b]) for k, x in features.items() if k.startswith(\'Q\')}\n\n\ndef fit_probe(x, y, fit_rows):\n    m = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000,\n        solver=\'lbfgs\', random_state=42))\n    with warnings.catch_warnings(), threadpool_limits(limits=1):\n        warnings.simplefilter(\'error\', ConvergenceWarning)\n        m.fit(x[fit_rows], np.asarray(y)[fit_rows])\n    return m\n\n\ndef _p1(model, x):\n    col = np.flatnonzero(model.classes_ == 1)\n    if len(col) != 1: raise ValueError(\'Binary probe is missing a class.\')\n    return model.predict_proba(x)[:, col[0]]\n\n\ndef run_cv(f, features, out_dir, *, specs=SPECS, save_models=False) -> dict[str, pd.DataFrame]:\n    """No test-dependent selection; all methods use exactly the E14 FIT partition."""\n    import joblib\n    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)\n    if not (out / \'analysis_plan.json\').exists(): raise RuntimeError(\'Freeze the analysis plan first.\')\n    if (out / \'oof_predictions.csv\').exists(): raise FileExistsError(\'Output exists. Use a new run directory.\')\n    locked = json.loads((out / \'analysis_plan.json\').read_text(encoding=\'utf-8\'))\n    if locked[\'methods\'] != json.loads(json.dumps([asdict(s) for s in specs])):\n        raise ValueError(\'Requested methods differ from the frozen plan.\')\n    if locked[\'seed\'] != 42 or locked[\'C_logreg\'] != 0.1:\n        raise ValueError(\'This implementation uses the fixed E14 seed and regularization.\')\n    validate_features(features, len(f))\n    missing = {k for s in specs for k in s.keys} - set(features)\n    if missing: raise KeyError(f\'Missing neutral features: {sorted(missing)}. No synthetic substitution.\')\n    matrices = {s.name: np.concatenate([features[k] for k in s.keys], axis=1)\n                for s in specs if s.mode == \'joint4\'}\n    pred = {s.name: np.full((len(f), 4), np.nan) for s in specs}\n    fold_map = f.question.map(build_outer_map(f)).to_numpy()\n    manifest = []; timings = []\n    for fold in range(5):\n        split = make_split(f, fold); fit = np.flatnonzero(split.role.eq(\'fit\')); test = np.flatnonzero(split.role.eq(\'test\'))\n        split.to_csv(out / f\'split_fold_{fold}.csv\', index=False)\n        # Fit identical binary reader only once per fold when reused across comparisons.\n        binary = {}\n        for spec in specs:\n            start = time.perf_counter()\n            if spec.mode == \'joint4\':\n                x = matrices[spec.name]; m = fit_probe(x, f.state, fit)\n                if list(m.classes_) != [0, 1, 2, 3]: raise ValueError(\'Wrong state order.\')\n                p = m.predict_proba(x[test]); saved = {\'model\': m, \'keys\': spec.keys, \'mode\': spec.mode}\n                dim = x.shape[1]\n            elif spec.mode == \'factor\':\n                models = []\n                for axis, key in zip([\'k\', \'c\'], spec.keys):\n                    if (axis, key) not in binary:\n                        binary[(axis, key)] = fit_probe(features[key], f[axis], fit)\n                    models.append(binary[(axis, key)])\n                p = factor_probs(_p1(models[0], features[spec.keys[0]][test]),\n                                 _p1(models[1], features[spec.keys[1]][test]))\n                saved = {\'models\': models, \'keys\': spec.keys, \'mode\': spec.mode}\n                dim = sum(features[k].shape[1] for k in set(spec.keys))\n            else: raise ValueError(spec.mode)\n            if not np.isnan(pred[spec.name][test]).all(): raise ValueError(\'Duplicate OOF prediction.\')\n            pred[spec.name][test] = p\n            if save_models: joblib.dump(saved, out / f\'fold_{fold}_{spec.name}.joblib\')\n            manifest.append({\'fold\': fold, \'method\': spec.name, \'n_fit_rows\': len(fit),\n                \'n_fit_questions\': f.iloc[fit].question.nunique(), \'n_test_rows\': len(test),\n                \'unique_input_dimensions\': dim, \'uses_probe_at_runtime\': True,\n                \'test_used_for_fit_or_selection\': False})\n            timings.append({\'fold\': fold, \'method\': spec.name, \'seconds\': time.perf_counter() - start})\n        print(f\'Fold {fold + 1}/5: all fixed readers predicted.\', flush=True)\n    frames = {}\n    for name, p in pred.items():\n        if not np.isfinite(p).all() or not np.allclose(p.sum(1), 1): raise ValueError(\'Incomplete probabilities.\')\n        d = f[[\'row\', \'question\', \'k\', \'c\', \'state\']].copy().rename(columns={\'state\': \'true_state\'})\n        d[\'fold\'] = fold_map; d[\'method\'] = name; d[\'pred_state\'] = p.argmax(1)\n        for j in range(4): d[f\'p{j}\'] = p[:, j]\n        frames[name] = d\n    pd.concat(frames.values(), ignore_index=True).to_csv(out / \'oof_predictions.csv\', index=False)\n    pd.DataFrame(manifest).to_csv(out / \'fit_manifest.csv\', index=False)\n    pd.DataFrame(timings).to_csv(out / \'fit_timing.csv\', index=False)\n    write_json(out / \'environment.json\', environment())\n    return frames\n\n\ndef metrics4(y, probs) -> dict:\n    y = np.asarray(y); p = np.asarray(probs); pred = p.argmax(1)\n    out = {\'four_macro_F1\': float(f1_score(y, pred, labels=[0,1,2,3], average=\'macro\', zero_division=0)),\n        \'four_BAcc\': float(balanced_accuracy_score(y, pred)), \'four_accuracy\': float(accuracy_score(y, pred)),\n        \'NLL\': float(log_loss(y, p, labels=[0,1,2,3])),\n        \'Brier4\': float(np.mean(np.sum((p - np.eye(4)[y]) ** 2, axis=1)))}\n    wrongs = {}\n    for axis, pos in [(\'K\', [0,1]), (\'C\', [0,3])]:\n        truth = np.isin(y, pos).astype(int); score = p[:,pos].sum(1)\n        out[axis + \'_AUROC\'] = float(roc_auc_score(truth, score))\n        out[axis + \'_BAcc\'] = float(balanced_accuracy_score(truth, score >= .5))\n        # Joint state errors use the joint argmax, not marginal threshold decisions.\n        wrongs[axis] = np.isin(pred, pos) != truth.astype(bool)\n    out[\'K_only_wrong\'] = float(np.mean(wrongs[\'K\'] & ~wrongs[\'C\']))\n    out[\'C_only_wrong\'] = float(np.mean(wrongs[\'C\'] & ~wrongs[\'K\']))\n    out[\'both_wrong\'] = float(np.mean(wrongs[\'C\'] & wrongs[\'K\']))\n    return out\n\n\ndef paired_bootstrap(a: pd.DataFrame, b: pd.DataFrame, n_boot=2000, seed=42) -> dict:\n    """Paired question-cluster bootstrap of fixed OOF predictions; no refitting."""\n    a = a.sort_values(\'row\').reset_index(drop=True); b = b.sort_values(\'row\').reset_index(drop=True)\n    for col in [\'row\', \'question\', \'true_state\', \'fold\']:\n        if not a[col].equals(b[col]): raise ValueError(f\'Paired comparison mismatch: {col}\')\n    group, unique = pd.factorize(a.question, sort=True); g = len(unique)\n    cm_a = np.zeros((g,16), np.int32); cm_b = cm_a.copy()\n    np.add.at(cm_a, (group, a.true_state.to_numpy()*4 + a.pred_state.to_numpy()), 1)\n    np.add.at(cm_b, (group, b.true_state.to_numpy()*4 + b.pred_state.to_numpy()), 1)\n    def mf(v):\n        m = v.reshape(-1,4,4); tp = np.diagonal(m, axis1=1, axis2=2)\n        den = m.sum(1) + m.sum(2)\n        return np.mean(np.divide(2*tp, den, out=np.zeros_like(tp,dtype=float), where=den>0), axis=1)\n    rng = np.random.default_rng(seed); diffs = []\n    for start in range(0, n_boot, 100):\n        draw = rng.integers(0,g,size=(min(100,n_boot-start),g))\n        diffs.extend((mf(cm_a[draw].sum(1)) - mf(cm_b[draw].sum(1))).tolist())\n    point = float(mf(cm_a.sum(0))[0] - mf(cm_b.sum(0))[0])\n    lo, hi = np.quantile(diffs, [.025,.975])\n    return {\'delta_macro_F1\': point, \'ci95_lo\': float(lo), \'ci95_hi\': float(hi),\n            \'n_questions\': g, \'bootstrap_replicates\': n_boot,\n            \'scope\': \'fixed_OOF_predictions_not_full_pipeline_or_new_dataset\'}\n\n\ndef summarize(frames, out_dir, n_boot=2000) -> tuple[pd.DataFrame, pd.DataFrame]:\n    out = Path(out_dir); summaries=[]; folds=[]; perclass=[]\n    for name, d in frames.items():\n        p = d[[f\'p{i}\' for i in range(4)]].to_numpy()\n        summaries.append({\'method\': name, **metrics4(d.true_state, p)})\n        for fold, sub in d.groupby(\'fold\'):\n            folds.append({\'method\': name, \'fold\': fold,\n                          **metrics4(sub.true_state, sub[[f\'p{i}\' for i in range(4)]].to_numpy())})\n        pr,re,fs,ns = precision_recall_fscore_support(d.true_state,d.pred_state,labels=[0,1,2,3],zero_division=0)\n        for i in range(4): perclass.append({\'method\':name,\'state\':STATE_NAMES[i],\n            \'precision\':pr[i],\'recall\':re[i],\'F1\':fs[i],\'n\':int(ns[i])})\n        pd.DataFrame(confusion_matrix(d.true_state,d.pred_state,labels=[0,1,2,3]),\n            index=STATE_NAMES,columns=STATE_NAMES).to_csv(out / f\'confusion_{name}.csv\')\n    sm = pd.DataFrame(summaries); fd = pd.DataFrame(folds)\n    fs = fd.groupby(\'method\').four_macro_F1.agg([\'mean\',\'std\']).rename(columns={\'mean\':\'fold_F1_mean\',\'std\':\'fold_F1_std\'})\n    sm = sm.merge(fs,left_on=\'method\',right_index=True)\n    rows=[]\n    for a,b,kind in CONTRASTS:\n        if a in frames and b in frames: rows.append({\'comparison\':kind,\'A\':a,\'B\':b,\n            **paired_bootstrap(frames[a],frames[b],n_boot=n_boot)})\n    contrast = pd.DataFrame(rows)\n    sm.to_csv(out/\'summary.csv\',index=False); fd.to_csv(out/\'fold_metrics.csv\',index=False)\n    pd.DataFrame(perclass).to_csv(out/\'per_state_metrics.csv\',index=False)\n    contrast.to_csv(out/\'paired_comparisons.csv\',index=False)\n    lines = [\'# E18A 실행 결과\', \'\', \'실험 성격: 기존 p8s 개발셋의 후속 OOF 진단. 독립 확증이 아님.\',\n             \'QA 생성/steering은 실행하지 않음. 아래 숫자는 답변 EM이 아니라 상태 분류 성능임.\', \'\',\n             \'|방법|4-state Macro-F1|K AUROC|C AUROC|\', \'|---|---:|---:|---:|\']\n    for _,r in sm.iterrows(): lines.append(f"|{r[\'method\']}|{r.four_macro_F1:.4f}|{r.K_AUROC:.4f}|{r.C_AUROC:.4f}|")\n    lines += [\'\', \'## Paired differences\', \'CI는 질문 단위 fixed-OOF bootstrap이며 학습/설계 선택 불확실성 전체를 포함하지 않음.\']\n    for _,r in contrast.iterrows(): lines.append(f"\\n{r.comparison}: {100*r.delta_macro_F1:+.2f} pp [{100*r.ci95_lo:+.2f}, {100*r.ci95_hi:+.2f}]")\n    if \'M2_L7lastK_L13lastC\' not in frames:\n        lines += [\'\', \'**L7 실험 미실행: 현재 결과로 early-K 가설의 성공/실패를 판단할 수 없음.**\']\n    lines += [\'\', \'해석: 첫 질문 위치는 Q-first causal mask로 뒤 문맥을 볼 수 없다. 이것은 특정 early layer만의 성질이 아니다.\',\n        \'좋은 state decoder가 좋은 QA router라는 보장은 없다. QA 검증은 별도 고정 actuator로 수행해야 한다.\',\n        \'이 결과를 보고 최고 OOF 방법을 선택해 독립 test 성능으로 제시하지 않는다.\']\n    (out/\'RESULTS_KO.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n    return sm, contrast\n\n\ndef audit_reference(predictions: pd.DataFrame, reference_path) -> dict:\n    a = predictions.sort_values(\'row\').reset_index(drop=True)\n    b = pd.read_csv(reference_path).sort_values(\'row\').reset_index(drop=True)\n    for col in [\'row\',\'fold\',\'true_state\']:\n        if not np.array_equal(a[col], b[col]): raise ValueError(f\'E14 reference mismatch: {col}\')\n    return {\'n\':len(a), \'same_predicted_state\':int(np.sum(a.pred_state==b.pred_state)),\n            \'max_probability_difference\':float(np.max(np.abs(a[[f\'p{i}\' for i in range(4)]].to_numpy()-\n                                                            b[[f\'p{i}\' for i in range(4)]].to_numpy()))),\n            \'reference_macro_F1\':float(f1_score(b.true_state,b.pred_state,average=\'macro\')),\n            \'rerun_macro_F1\':float(f1_score(a.true_state,a.pred_state,average=\'macro\')),\n            \'environment\':environment()}\n', 'e18a_features.py': '"""Read-only Drive retrieval and neutral prefill feature capture for E18A."""\nfrom __future__ import annotations\nimport gc\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport numpy as np\nfrom e18a_core import (MODEL_ID, MODEL_REVISION, SOURCE_FILES, render_prompt,\n                      sha256_file, write_json, environment)\n\n\ndef obtain_sources(work, search_roots=()):\n    """Prefer exact-hash local files. Download from the user\'s authenticated Drive otherwise."""\n    import shutil\n    work = Path(work); work.mkdir(parents=True, exist_ok=True)\n    service = None; paths = {}\n    for key, spec in SOURCE_FILES.items():\n        target = work / spec[\'name\']\n        candidates = [target] + [Path(p) / spec[\'name\'] for p in search_roots]\n        match = next((p for p in candidates if p.is_file() and sha256_file(p) == spec[\'sha256\']), None)\n        if match is not None:\n            if match.resolve() != target.resolve(): shutil.copyfile(match, target)\n        else:\n            if service is None:\n                try:\n                    from google.colab import auth\n                    import google.auth\n                    from googleapiclient.discovery import build\n                    auth.authenticate_user()\n                    creds, _ = google.auth.default()\n                    service = build(\'drive\', \'v3\', credentials=creds, cache_discovery=False)\n                except ImportError as e:\n                    raise RuntimeError(f\'Put the exact input files in {work}, or run in Colab with Drive access.\') from e\n            from googleapiclient.http import MediaIoBaseDownload\n            tmp = target.with_name(target.name + \'.download\')\n            try:\n                req = service.files().get_media(fileId=spec[\'id\'])\n                with tmp.open(\'wb\') as fh:\n                    dl = MediaIoBaseDownload(fh, req); done = False\n                    while not done: _, done = dl.next_chunk()\n                if sha256_file(tmp) != spec[\'sha256\']:\n                    raise ValueError(f\'Source changed: {spec["name"]}. Do not silently adopt a different version.\')\n                tmp.replace(target)\n            finally:\n                tmp.unlink(missing_ok=True)\n        if sha256_file(target) != spec[\'sha256\']: raise ValueError(f\'Checksum mismatch: {key}\')\n        paths[key] = target\n    return paths\n\n\nclass BlockCapture:\n    """Observe zero-based block outputs and L13 o_proj inputs; never modify tensors."""\n    def __init__(self, model, qidx):\n        self.model = model; self.qidx = qidx; self.handles = []; self.cache = {}; self.counts = {}\n\n    def __enter__(self):\n        import torch\n        def make_post(layer):\n            def hook(mod, args, out):\n                h = out[0] if isinstance(out, tuple) else out\n                if h.ndim != 3: raise RuntimeError(\'Unexpected decoder block output.\')\n                self.counts[layer] = self.counts.get(layer, 0) + 1\n                b = torch.arange(len(h), device=h.device)\n                self.cache[f\'Q{layer}\'] = h[b, self.qidx].detach().float().cpu().numpy().astype(np.float16)\n                self.cache[f\'L{layer}\'] = h[:, -1].detach().float().cpu().numpy().astype(np.float16)\n            return hook\n        for layer in [7, 13]:\n            self.handles.append(self.model.model.layers[layer].register_forward_hook(make_post(layer)))\n        def oproj_pre(mod, args):\n            x = args[0]; nheads = self.model.config.num_attention_heads\n            hv = x.reshape(x.shape[0], x.shape[1], nheads, -1)\n            b = torch.arange(len(x), device=x.device)\n            self.cache[\'HEAD_Q13\'] = hv[b, self.qidx].detach().float().cpu().numpy().astype(np.float16)\n            self.cache[\'HEAD_LAST13\'] = hv[:, -1].detach().float().cpu().numpy().astype(np.float16)\n        self.handles.append(self.model.model.layers[13].self_attn.o_proj.register_forward_pre_hook(oproj_pre))\n        return self\n\n    def __exit__(self, *args):\n        for h in self.handles: h.remove()\n        self.handles.clear()\n\n\ndef collect_neutral(model, tokenizer, prompts, out_dir, batch_size=4, max_tokens=4096):\n    """One standard neutral prefill per batch. No CB, no steering, no labels, no decode."""\n    import torch\n    if set(prompts.columns) != {\'row\', \'question\', \'ctx_text\'}:\n        raise ValueError(\'Only row/question/context may enter the capture function; do not pass gold or labels.\')\n    if not np.array_equal(prompts.row, np.arange(len(prompts))): raise ValueError(\'Rows must be canonical.\')\n    if getattr(model.config, \'_commit_hash\', None) != MODEL_REVISION:\n        raise ValueError(\'Model revision is not the frozen original E14 revision.\')\n    if next(model.parameters()).dtype != torch.bfloat16 or not next(model.parameters()).is_cuda:\n        raise ValueError(\'Fresh capture requires the BF16 model on CUDA; use cache-only mode otherwise.\')\n    if model.config.hidden_size != 4096 or model.config.num_attention_heads != 32:\n        raise ValueError(\'Expected original Llama-3.1-8B architecture.\')\n    if batch_size != 4: raise ValueError(\'Use a NEW study to change the frozen batch size.\')\n    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)\n    p = [render_prompt(tokenizer, str(r.question), str(r.ctx_text)) for r in prompts.itertuples()]\n    if max(map(lambda d: len(d[\'ids\']), p)) > max_tokens:\n        raise ValueError(\'Prompt exceeds length limit; no silent truncation.\')\n    manifest = {\'model_id\': MODEL_ID, \'revision\': MODEL_REVISION, \'dtype\': \'bfloat16\',\n        \'batch_size\': batch_size, \'prompt_hashes\': [d[\'prompt_sha256\'] for d in p],\n        \'neutral\': True, \'decode\': False, \'packed_CB\': False, \'layer_indexing\': \'zero_based_block_output\',\n        \'environment\': environment()}\n    mp = out / \'capture_manifest.json\'\n    if mp.exists() and json.loads(mp.read_text()) != manifest:\n        raise ValueError(\'Capture manifest/environment changed. Use a new feature directory.\')\n    if not mp.exists(): write_json(mp, manifest)\n    dev = next(model.parameters()).device\n    model.eval(); chunks = []\n    with torch.inference_mode():\n        for st in range(0, len(prompts), batch_size):\n            end = min(st + batch_size, len(prompts)); rows = np.arange(st, end)\n            path = out / f\'batch_{st:06d}.npz\'\n            if path.exists():\n                with np.load(path, allow_pickle=False) as z: chunk = {k: z[k].copy() for k in z.files}\n                if not np.array_equal(chunk[\'rows\'], rows): raise ValueError(\'Cached rows differ.\')\n                required = {\'Q7\', \'L7\', \'Q13\', \'L13\', \'HEAD_Q13\', \'HEAD_LAST13\'}\n                if not required.issubset(chunk): raise ValueError(\'Incomplete capture shard.\')\n            else:\n                pp = p[st:end]; t = max(len(x[\'ids\']) for x in pp)\n                ids = torch.full((len(pp),t),tokenizer.pad_token_id,dtype=torch.long,device=dev)\n                mask = torch.zeros_like(ids); qi = []\n                for j, x in enumerate(pp):\n                    off = t - len(x[\'ids\']); ids[j, off:] = torch.tensor(x[\'ids\'], device=dev)\n                    mask[j, off:] = 1; qi.append(off + x[\'qpos\'])\n                pos = (mask.cumsum(-1) - 1).clamp_min(0)\n                qidx = torch.tensor(qi, device=dev)\n                try:\n                    with BlockCapture(model, qidx) as cap:\n                        res = model.model(input_ids=ids, attention_mask=mask, position_ids=pos,\n                                          use_cache=False, return_dict=True)\n                    del res\n                    if cap.counts != {7:1,13:1}: raise RuntimeError(\'Expected exactly one read per block per prefill.\')\n                    chunk = {\'rows\': rows, **cap.cache}\n                    tmp = path.with_name(path.name + \'.tmp\')\n                    with tmp.open(\'wb\') as fh: np.savez_compressed(fh, **chunk)\n                    tmp.replace(path)\n                except torch.cuda.OutOfMemoryError as e:\n                    raise RuntimeError(\'GPU OOM. Shards saved. Restart on a larger GPU; no automatic quantization.\') from e\n                finally:\n                    del ids, mask, pos, qidx\n            chunks.append(chunk)\n            if st % 80 == 0: print(f\'Neutral prefill {end}/{len(prompts)} rows\', flush=True)\n    combined = {k: np.concatenate([c[k] for c in chunks], axis=0) for k in chunks[0]}\n    with (out / \'neutral_features_all.npz\').open(\'wb\') as fh: np.savez_compressed(fh, **combined)\n    features = {k: combined[k].astype(np.float32) for k in [\'Q7\',\'L7\',\'Q13\',\'L13\']}\n    return features, {\'source\': \'fresh_E18A_neutral_prefill\', \'manifest_sha256\': sha256_file(mp),\n                      \'bank_sha256\': sha256_file(out / \'neutral_features_all.npz\')}\n', 'test_e18a.py': "from pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport pytest\nfrom e18a_core import *\n\n\ndef toy_frame(n=100):\n    q = np.repeat([f'q{i:03}' for i in range(n)],2)\n    k = np.repeat(np.arange(n)%2,2); c = np.tile([0,1],n)\n    return pd.DataFrame({'row':np.arange(2*n),'question':q,'k':k,'c':c,'state':cell_ids(k,c)})\n\n\ndef test_state_order():\n    assert cell_ids([1,1,0,0],[1,0,0,1]).tolist() == [0,1,2,3]\n\n\ndef test_product_probabilities():\n    p = factor_probs(np.array([.7,.3]),np.array([.8,.2]))\n    np.testing.assert_allclose(p.sum(1),1)\n    np.testing.assert_allclose(p[:,[0,1]].sum(1),[.7,.3])\n    np.testing.assert_allclose(p[:,[0,3]].sum(1),[.8,.2])\n\n\ndef test_bad_probabilities():\n    with pytest.raises(ValueError): factor_probs([2], [.5])\n    with pytest.raises(ValueError): factor_probs([np.nan],[.5])\n\n\ndef test_question_groups_and_oof_coverage():\n    f=toy_frame(); seen=np.zeros(len(f),int)\n    for fold in range(5):\n        s=make_split(f,fold)\n        assert s.groupby('question').role.nunique().eq(1).all()\n        fit=set(s.loc[s.role.eq('fit'),'question']); test=set(s.loc[s.role.eq('test'),'question'])\n        assert fit.isdisjoint(test)\n        seen[s.role.eq('test')]+=1\n    np.testing.assert_array_equal(seen,1)\n\n\ndef test_scaler_not_fit_on_test():\n    x=np.arange(80,dtype=float).reshape(40,2); y=np.tile([0,1],20); fit=np.arange(20)\n    m=fit_probe(x,y,fit)\n    np.testing.assert_allclose(m.named_steps['standardscaler'].mean_,x[fit].mean(0))\n\n\ndef test_feature_validation():\n    with pytest.raises(ValueError): validate_features({'Q7':np.zeros((2,3,4))},2)\n    with pytest.raises(ValueError): validate_features({'Q7':np.array([[np.nan]])},1)\n\n\ndef test_plan_cannot_change(tmp_path):\n    freeze_plan(tmp_path,bootstrap=30)\n    freeze_plan(tmp_path,bootstrap=30)\n    with pytest.raises(ValueError): freeze_plan(tmp_path,bootstrap=31)\n\n\ndef test_small_end_to_end_and_bootstrap(tmp_path):\n    f=toy_frame(); rng=np.random.default_rng(21)\n    x=rng.normal(size=(len(f),12)); x[:,0]+=f.k*1.7; x[:,1]+=f.c*1.5\n    features={'L13':x}; specs=SPECS[:2]\n    freeze_plan(tmp_path,specs=specs,bootstrap=30)\n    frames=run_cv(f,features,tmp_path,specs=specs)\n    assert len(frames['M0_L13_joint4'])==len(f)\n    z=paired_bootstrap(frames['M0_L13_joint4'],frames['M0_L13_joint4'],n_boot=30)\n    assert z['delta_macro_F1']==0 and z['ci95_lo']==0 and z['ci95_hi']==0\n    sm,comp=summarize(frames,tmp_path,n_boot=30)\n    assert len(sm)==2 and len(comp)==1\n    with pytest.raises(FileExistsError): run_cv(f,features,tmp_path,specs=specs)\n\n\ndef test_no_synthetic_substitution(tmp_path):\n    f=toy_frame(); freeze_plan(tmp_path)\n    with pytest.raises(KeyError): run_cv(f,{'L13':np.ones((len(f),2))},tmp_path)\n\n\ndef test_invariance():\n    f=toy_frame(); x=np.repeat(np.arange(100)[:,None],2,axis=0).astype(float)+1\n    d=question_invariance(f,{'Q7':x,'L13':x})\n    assert list(d)==['Q7']; assert d['Q7']['max_relative_L2']==0\n\n\ndef test_paired_alignment_required():\n    f=toy_frame(); d=f.rename(columns={'state':'true_state'}); d['fold']=0; d['pred_state']=d.true_state\n    b=d.copy(); b.loc[0,'true_state']=1\n    with pytest.raises(ValueError): paired_bootstrap(d,b,n_boot=10)\n\n\ndef test_capture_hooks_observe_not_modify():\n    import torch\n    from types import SimpleNamespace\n    from e18a_features import BlockCapture\n    class Block(torch.nn.Module):\n        def __init__(self):\n            super().__init__(); self.self_attn=torch.nn.Module(); self.self_attn.o_proj=torch.nn.Linear(8,8,bias=False)\n        def forward(self,x): return x+self.self_attn.o_proj(x)\n    class Backbone(torch.nn.Module):\n        def __init__(self): super().__init__(); self.layers=torch.nn.ModuleList([Block() for _ in range(14)])\n        def forward(self,x):\n            for layer in self.layers: x=layer(x)\n            return x\n    model=torch.nn.Module(); model.model=Backbone(); model.config=SimpleNamespace(num_attention_heads=2)\n    x=torch.randn(2,5,8); qi=torch.tensor([1,2]); before=model.model(x)\n    with BlockCapture(model,qi) as cap: after=model.model(x)\n    assert torch.equal(before,after)\n    assert cap.counts=={7:1,13:1}; assert cap.cache['Q7'].shape==(2,8)\n    assert cap.cache['HEAD_Q13'].shape==(2,2,4)\n    assert not cap.handles\n"}
for name, source in _MODULE_SOURCES.items():
    (MODULE_DIR / name).write_text(source, encoding='utf-8')
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
from e18a_core import *
from e18a_features import obtain_sources, collect_neutral
print('E18A implementation loaded.')


In [ ]:
# 짧은 합성/단위 테스트: 이 출력은 연구 성능 수치가 아닙니다.
# split 누수, 상태 순서, scaler fit 범위, bootstrap 정렬, observer hook을 검사합니다.
subprocess.check_call([sys.executable, '-m', 'pytest', '-q', str(MODULE_DIR / 'test_e18a.py')], cwd=MODULE_DIR)

## 3. 원본 데이터 / label / split 검증
K 정의는 E14의 `kside_corr`로 고정한다. E17의 `K_strict`와 섞지 않는다. 이 E14 입력은 612질문 중 K+362, K−250이다.

`row`·4-state 정답·outer fold가 저장된 E14 OOF 결과와 **모두 일치해야** 진행한다. 같은 질문의 C+/C− 두 행은 항상 함께 분할한다.

내부 fit 비율도 기존 E14와 같게 약 60%로 유지한다. 이번 방법만 80%에 학습시키고 기존 60% 결과와 비교하지 않는다.

In [ ]:
# E14-compatible source retrieval; keep all fixed source hashes unchanged.
import subprocess, shutil
from e18a_core import SOURCE_FILES
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'gdown'])
import gdown
WORK = Path('/content/e18a_inputs') if IN_COLAB else Path('/mnt/data')
WORK.mkdir(parents=True, exist_ok=True)
for key, spec in SOURCE_FILES.items():
    target = WORK / spec['name']
    candidates = [target, E14_ROOT / spec['name'], PROJECT_ROOT / spec['name']]
    local = next((p for p in candidates if p.is_file() and sha256_file(p) == spec['sha256']), None)
    if local is not None:
        if local != target:
            shutil.copy2(local, target)
    else:
        downloaded = gdown.download(id=spec['id'], output=str(target) + '.download', quiet=False)
        if downloaded is None:
            raise RuntimeError('Source download failed: ' + key)
        assert sha256_file(Path(downloaded)) == spec['sha256'], 'Source hash mismatch: ' + key
        Path(downloaded).replace(target)
    assert sha256_file(target) == spec['sha256']
    print('Verified source:', key, target.name)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
WORK = (Path('/content/e18a_inputs') if IN_COLAB else Path('/mnt/data'))
PATHS = obtain_sources(WORK, search_roots=[E14_ROOT, PROJECT_ROOT])
frame = load_labels(PATHS['behavior'], PATHS['oof13'])
SPECS_RUN = SPECS if MODE == 'full' else SPECS[:2]
OUT = PROJECT_ROOT / (RUN_ID + ('_l13_only' if MODE == 'l13_only' else ''))
freeze_plan(OUT, specs=SPECS_RUN, bootstrap=BOOTSTRAP_REPS)
write_json(OUT / 'source_hashes.json', {k: sha256_file(p) for k,p in PATHS.items()})
write_json(OUT / 'implementation_hashes.json', {p.name:sha256_file(p) for p in MODULE_DIR.glob('*.py')})
print('Rows:',len(frame), '| Questions:',frame.question.nunique())
print('K+ questions:',int(frame.groupby('question').k.first().sum()))
display(frame.groupby(['k','c']).size().rename('n').reset_index())
for fold in range(5):
    sp = make_split(frame, fold)
    sp.to_csv(OUT / f'split_fold_{fold}.csv', index=False)
    print('Fold',fold,sp.role.value_counts().to_dict())
print('Analysis lock:',OUT / 'analysis_plan.json')

## 4. 중립 표현 확보
E11 cache에는 `Q7/L7/Q13/L13`이 저장되어 있다. `Q`는 질문 마지막 토큰, `L`은 프롬프트 마지막 토큰이다.

**Q-first의 질문 위치는 뒤 문맥을 보지 못한다. 이는 L7뿐 아니라 L13 질문 위치에도 동일하다.** 따라서 “early라서 문맥에 오염되지 않았다”라고 단정하지 않는다. 질문 위치 표현을 읽는 것은 closed-book 답을 추가 생성하는 것과도 다르다.

캐시는 각 행의 원래 prompt SHA, noop 여부, 개입량 0, row 순서를 확인한 뒤 사용한다. 새로 수집할 때는 `row/question/context`만 모델 함수에 넘긴다.

In [ ]:
L13_ARCHIVE = load_e14_block_bank(PATHS['bank13'], len(frame))
features = None
provenance = None
if MODE == 'l13_only':
    features = L13_ARCHIVE
    provenance = {'source':'archived_E14_BLOCK_LAST','early_layer_experiment_executed':False}
else:
    import transformers, torch
    if transformers.__version__ != '4.57.1':
        raise RuntimeError('Transformers 4.57.1이 필요합니다. 설치 후 세션 재시작.')
    from huggingface_hub import get_token, login
    if not get_token():
        login(add_to_git_credential=False)
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'
    use_e11 = FEATURE_SOURCE == 'e11_cache' or (
        FEATURE_SOURCE == 'auto' and (E11_NEUTRAL / 'manifest.json').exists())
    if use_e11:
        features, provenance = load_e11_neutral(E11_NEUTRAL, frame, tokenizer=tokenizer)
        print('완전한 E11 neutral cache를 재사용했습니다. 추가 LLM 추론 없음.')
    else:
        if not torch.cuda.is_available():
            raise RuntimeError('L7/Q7/Q13 전체 cache가 없습니다. GPU 런타임 또는 정확한 E11 cache 경로가 필요합니다.')
        if 'model' in globals():
            del model
        gc.collect(); torch.cuda.empty_cache()
        free,total = torch.cuda.mem_get_info()
        print('GPU:', torch.cuda.get_device_name(0), '| free GiB:',round(free/1024**3,1))
        if free < 20*1024**3:
            raise RuntimeError('BF16 8B prefill을 위한 여유 메모리가 부족합니다. 더 큰 GPU/새 세션을 사용하세요.')
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
            torch_dtype=torch.bfloat16, device_map={'':'cuda:0'}, attn_implementation='sdpa',
            trust_remote_code=False).eval()
        features, provenance = collect_neutral(model, tokenizer,
            frame[['row','question','ctx_text']].copy(), OUT / 'neutral_capture', batch_size=BATCH_SIZE)
        del model
        gc.collect(); torch.cuda.empty_cache()
write_json(OUT / 'feature_provenance.json', provenance)
validate_features(features, len(frame))
print({k:v.shape for k,v in features.items()})

In [ ]:
# 모든 방법은 features라는 동일 bank를 사용한다. 다른 run의 early/late를 섞지 않는다.
agreement = feature_agreement(features['L13'], L13_ARCHIVE['L13'])
write_json(OUT / 'L13_feature_agreement.json', agreement)
print('L13 vs E14 archive:',agreement)
if agreement['max_relative_L2'] > 0.02:
    raise RuntimeError('E14와 L13 표현 차이가 사전 경보 한도 2%를 넘었습니다. '
                       '프롬프트/모델/캐시를 확인하세요. 다른 RUN_ID의 fresh 수집은 가능하지만 정확 재현으로 부르면 안 됩니다.')
inv = question_invariance(frame, features)
write_json(OUT / 'question_prefix_invariance.json', inv)
print('동일 질문 C+/C−의 question-position 일치도:', inv)
print('이 검사는 bitwise 동일성을 보장하는 것이 아니라 padding/bf16 차이를 포함한 관찰값입니다.')

## 5. 고정 5-fold 실행
모든 조건을 **동일한 fit row / 정규화 / LogisticRegression C=0.1 / seed42**로 학습한다. K/C factorized 방식은 최종 단계까지 확률을 유지하고, 아래 순서로 합친다.

$$p = [p_Kp_C,\ p_K(1-p_C),\ (1-p_K)(1-p_C),\ (1-p_K)p_C].$$

이 곱은 conditional independence를 가정하는 간단한 readout이다. 진짜 joint 확률을 보장하지 않으므로 직접 joint4 방식과 비교한다. 학습은 supervised이며, runtime에도 probe를 쓴다. early-K가 더 좋을 것이라고 미리 결론내리지 않는다.

In [ ]:
# 오래 걸리는 중립 LLM 수집과 달리, 이 단계는 저장 feature의 CPU probe 학습이다.
OOF = run_cv(frame, features, OUT, specs=SPECS_RUN, save_models=SAVE_MODELS)
AUDIT = audit_reference(OOF['M0_L13_joint4'], PATHS['oof13'])
write_json(OUT / 'baseline_reproduction_audit.json', AUDIT)
print('기존 E14와 같은 predicted state:', AUDIT['same_predicted_state'], '/', AUDIT['n'])
print('기존 저장 Macro-F1:', AUDIT['reference_macro_F1'])
print('이번 동일 bank/환경 기준선 Macro-F1:', AUDIT['rerun_macro_F1'])
if AUDIT['same_predicted_state'] != AUDIT['n']:
    print('주의: 정확 재현은 아닙니다. 차이를 숨기지 말고 새 방법은 이번 재실행 기준선과 비교하세요.')

## 6. 효과 판정
주 지표는 pooled OOF 4-state Macro-F1이다. C+/C−가 한 질문 쌍이므로 두 행을 묶어 paired bootstrap한다.

**CI에 0이 포함되면 개선이 확증됐다고 하지 않는다.** CI가 양수라고 해도 같은 개발셋을 재사용한 후속 진단이다. 이 bootstrap은 이미 학습된 OOF 예측을 조건으로 하며 학습/연구 설계 전체 불확실성을 포함하지 않는다.

같은 Macro-F1이어도 K+C−, K−C+에서 어떤 오류가 줄었는지 다를 수 있다. 상태별 recall과 K/C 오류 분해를 함께 확인한다.

In [ ]:
SUMMARY, COMPARISONS = summarize(OOF, OUT, n_boot=BOOTSTRAP_REPS)
display(SUMMARY)
display(COMPARISONS)
if MODE == 'full':
    primary = COMPARISONS.query("comparison == 'PRIMARY_layer_only'").iloc[0]
    print('주 비교 M2−M1:',f"{100*primary.delta_macro_F1:+.2f} pp",
          f"95% CI [{100*primary.ci95_lo:+.2f}, {100*primary.ci95_hi:+.2f}]")
    if primary.ci95_lo > 0:
        print('이 개발셋 OOF에서 early-last K의 개선 증거가 관찰됨. 외부 확증/QA 이득은 별개.')
    elif primary.ci95_hi < 0:
        print('이 개발셋에서는 early-last K가 오히려 나쁨. early-K 가설을 성공으로 보고하면 안 됨.')
    else:
        print('early-last K의 개선 여부가 불확실함. 단순 점수차만으로 성공 판정하지 않음.')
else:
    print('L13-only 진단입니다. early/late 주 실험은 실행하지 않았습니다.')
display(pd.read_csv(OUT / 'per_state_metrics.csv'))

## 7. 연구 스토리 연결
**판별이 개선됨 → steering도 개선됨**은 성립하지 않는다. 이후에는 가장 먼저 actuator를 고정하고 기존/새 router만 바꾸는 QA 실험을 해야 한다. 그때 Recovery/Damage, 전체 EM, C+ 안전성을 따로 본다.

probe 없이 쓰려는 최종 방향은 다음 단계다. 지금 결과가 유망하면 같은 위치/레이어 정보를 소수 head의 벡터로 제한하고 fixed geometry readout과 비교한다. 거리/프로토타입 규칙도 분류 규칙이므로 ‘분류 자체를 없앴다’고 주장하지 않는다.

L13 상태를 읽고 개입한다면 쓰기 위치는 L14 이후여야 한다. L13을 보고 L8로 돌아가 개입하면 현재의 one-prefill 구조가 아니다.

In [ ]:
# 결과만 묶는다. 8B 모델/원본 대용량 activation은 zip에 넣지 않는다.
from zipfile import ZipFile, ZIP_DEFLATED
ZIP_PATH = OUT.parent / (OUT.name + '_results.zip')
with ZipFile(ZIP_PATH, 'w', ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and 'neutral_capture' not in p.parts and p.suffix != '.joblib':
            z.write(p, p.relative_to(OUT))
print('결과 폴더:', OUT)
print('결과 zip:', ZIP_PATH)
from IPython.display import display, FileLink, Markdown
display(Markdown((OUT / 'RESULTS_KO.md').read_text(encoding='utf-8')))
display(FileLink(str(ZIP_PATH)))

## 참고와 실행 한계
원본은 E11 `E11_Causal_KC_OnePass_Overnight.ipynb`, E14 `E14_L13_HeadDissection_5Fold_plus_router_metrics.ipynb`, E14 L13 neutral bank 및 OOF CSV다. 소스 ID/해시는 내장 코드와 결과 manifest에 기록한다.

라이브러리 구현 참고: [scikit-learn common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html), [nested-vs-non-nested CV](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html), [Transformers 4.57.1 Llama](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/llama).

**제작 시 확인:** 12개 단위/합성 테스트와 실제 L13 M0/M1 CPU 실행. GPU에서의 8B fresh 수집 및 L7 포함 7조건 전체 실행은 이 파일을 제작한 환경에서 수행하지 않았다. 코드와 체크를 제공한 것이며 성능 향상을 미리 보장하지 않는다.